# tool 规范

## Function calling 为什么需要 structured Outputs？

默认情况下，当您使用函数调用时，API 将为您的参数提供匹配的数据。
但是在使用复杂的fucntion时，比如参数很复杂，模型可能偶尔会遗漏参数或弄错它们的类型。
结构化输出可以确保函数调用的模型输出将完全匹配您提供的架构。

## tools 参数

tools可以在每个API请求的参数中设置功能。

函数由其模式定义，该模式告知模型其功能以及所需的输入参数。它包含以下字段：

| 场地 | 描述 |
| :--- | :--- |
| type | 这应该始终function |
| name | 函数的名称（例如get＿weather） |
| description | 有关何时以及如何使用该功能的详细信息 |
| parameters | 定义函数输入参数的[JSON 模式](https://json-schema.org/) |
| strict | 是否对函数调用强制执行严格模式 |

实例：
```json
{
    "type": "function",
    "function": {
        "name": "get_weather",
        "description": "Retrieves current weather for the given location.",
        "parameters": {
            "type": "object",
            "properties": {
                "location": {
                    "type": "string",
                    "description": "City and country e.g. Bogotá, Colombia"
                },
                "units": {
                    "type": "string",
                    "enum": [
                        "celsius",
                        "fahrenheit"
                    ],
                    "description": "Units the temperature will be returned in."
                }
            },
            "required": [
                "location",
                "units"
            ],
            "additionalProperties": false
        },
        "strict": true
    }
}
```
因为它们parameters是由JSON 模式定义的，所以您可以利用它的许多丰富的功能，如属性类型、枚举、描述、嵌套对象和递归对象。

##  如何使用 structured Outputs
启用函数调用的结构化输出，只需设置 strict: true 即可, 比如下面的例子：
```json
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "strict": True,
            "parameters": {
                "type": "object",
                "properties": {
                    "location": {"type": "string"},
                    "unit": {"type": "string", "enum": ["c", "f"]},
                },
                "required": ["location", "unit"],
                "additionalProperties": False,
            },
        },
    }
]

```

当你设置 strict：true 之后，OpenAI API 将在您的第一次请求时预处理您提供的架构，并使用此产物来限制模型遵循您的架构。

模型将始终遵循您的确切架构，除了在以下几种情况下：

- 当模型的响应被截断时（可能是由于 max_tokens、停止标记或最大上下文长度）
- 当模型拒绝执行时
- 当有内容过滤器完成原因时

> 请注意，当您首次使用结构化输出发送新架构的请求时，由于需要处理架构，会有额外的延迟，但后续请求应该不会产生任何额外开销。



## 处理函数调用

当模型调用函数时，您必须执行该函数并返回结果。由于模型响应可能包含零个、一个或多个调用，因此最佳做法是假设存在多个调用。

响应数组包含type为function_call 的output条目。每个条目包含一个call_id（稍后用于提交函数结果）、name和arguments JSON 编码的。


```json

[
    {
        "id": "fc_12345xyz",
        "call_id": "call_12345xyz",
        "type": "function_call",
        "name": "get_weather",
        "arguments": "{\"location\":\"Paris, France\"}"
    },
    {
        "id": "fc_67890abc",
        "call_id": "call_67890abc",
        "type": "function_call",
        "name": "get_weather",
        "arguments": "{\"location\":\"Bogotá, Colombia\"}"
    },
    {
        "id": "fc_99999def",
        "call_id": "call_99999def",
        "type": "function_call",
        "name": "send_email",
        "arguments": "{\"to\":\"bob@email.com\",\"body\":\"Hi bob\"}"
    }
]
```

## 你可以对function calling的行为进行哪些设置？
- 设置并行调用function（也就是在一个response时，可以返回多个function calls）
比如，你希望了解三个城市的天气预报，通过并行函数调用，一次response就可以获取需要执行的函数
处理完整之后，可以将三个城市的结果，反馈给LLM
具体代码，请参考：https://platform.openai.com/docs/guides/function-calling


- 函数并行调用与结构化输出冲突时
在使用函数并行时，模型可能不能很好的最从 strict的设置，此时可以通过设置 parallel_tool_calls: false 的方式关闭函数并行调用。


- 通过 tool_choice 来控制是否调用function call
    - tool_choice 有三种设置：
        - auto： 由模型自己控制是否调用function
        - requried：模型总是调用function
        - none：不调用function      
            指定调用的function：例如 tool_choice: {"type": "function", "function": {"name": "my_function"}}